# M2 - Multimodal RAG Embedding Extraction (Kaggle Cloud)

1. Create a **New Notebook** on Kaggle.
2. Add the **H&M Personalized Fashion Recommendations** dataset to the notebook.
3. Go to Session Options -> Accelerator -> **Turn on GPU T4x2**.
4. Upload this file via File -> Import Notebook and Run All.

In [ ]:
!pip install faiss-gpu open_clip_torch

import os
import pandas as pd
import torch
from PIL import Image
import open_clip
import faiss
import numpy as np
from tqdm.auto import tqdm

## 1. Setup Data Paths and Model

In [ ]:
DATA_DIR = '../input/h-and-m-personalized-fashion-recommendations'

# Define device and load CLIP Model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
model = model.to(device)
model.eval()

## 2. Process Dataset into Batches
We'll sample 5000 images (or all) due to time limits. Change `NUM_SAMPLES` to `None` for the full dataset!

In [ ]:
articles_df = pd.read_csv(f'{DATA_DIR}/articles.csv')

NUM_SAMPLES = 5000
if NUM_SAMPLES:
    articles_df = articles_df.sample(n=NUM_SAMPLES, random_state=42)

article_ids = []
images = []

print('Validating image paths...')
for idx, row in tqdm(articles_df.iterrows(), total=len(articles_df)):
    article_id = str(row['article_id']).zfill(10)
    folder = article_id[:3]
    path = f'{DATA_DIR}/images/{folder}/{article_id}.jpg'
    if os.path.exists(path):
        article_ids.append(article_id)
        images.append(path)

## 3. Extract Embeddings using CLIP

In [ ]:
BATCH_SIZE = 256
all_embeddings = []

print('Running CLIP embeddings...')
with torch.no_grad():
    for i in tqdm(range(0, len(images), BATCH_SIZE)):
        batch_paths = images[i:i + BATCH_SIZE]
        batch_images = []
        for path in batch_paths:
            img = Image.open(path).convert('RGB')
            batch_images.append(preprocess(img))
        
        image_tensor = torch.stack(batch_images).to(device)
        image_features = model.encode_image(image_tensor)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        all_embeddings.append(image_features.cpu().numpy())

embeddings = np.vstack(all_embeddings).astype('float32')
print(f'Embeddings Shape: {embeddings.shape}')

## 4. Build FAISS Index and Export

In [ ]:
D = embeddings.shape[1] # Output dimensionality
index = faiss.IndexFlatIP(D) # Inner product for normalized vectors (Cosine Similarity)
index.add(embeddings)

# Save Index
faiss.write_index(index, 'm2_clip_faiss.bin')

# Save mapping
pd.DataFrame({'article_id': article_ids}).to_csv('m2_faiss_mapping.csv', index=False)

print('Successfully Exported! Download m2_clip_faiss.bin and m2_faiss_mapping.csv from the Output folder on the right panel.')